Automatically Detect Substations in pandapower networks
=========

This notebook demonstrates how to automatically reconstruct substations by detecting
the typical switch configuration of a substation.
If other variations of substation structure exist (e.g. additional CB switches modeled), the code would need to be
slightly modified to accommodate the variation.

In the following notebook, a small network is created, and substations are added (and a substation DataFrame created).
Then, a series of functions are called to reconstruct the substation DataFrame using only the information contained in
the net (e.g. bus, line, load, switch DataFrames) -- NOT consulting the existing substation DataFrame.
The entire substation detection procedure can be performed with the following function call:

```
df_final = detect_substations(net)
```

but the separate steps are called individually to give the user a chance to view the DataFrames at various intermediate steps.

In [1]:
import pandapower as pp
from pandapower.networks import case2869pegase, case118, case14
import pandapower.plotting as plot

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import copy

In [2]:
from pandapower_env.substation.create_double_busbar_substation import (
    create_double_busbar_substation,
    can_convert_to_n_busbar_substation,
    create_3bb_with_pst_substation,
)

from pandapower_env.substation.double_busbar_substation import (
    reorder_substation_df,
)

from pandapower_env.substation.detect_substations import (
    get_all_element_terminals_df,
    add_switch_info_to_elements_df,
    pivot_to_connected_buses,
    add_bus_coupler_columns,
    detect_substations,
)

In [3]:
# net = case2869pegase()
net = case14()

Create the substations
======

In [4]:
bus_indices = net.bus.index[2:]
create_3bb_with_pst_substation(net, 1)
for bus in bus_indices:
    if not can_convert_to_n_busbar_substation(net, bus):
        continue
    create_double_busbar_substation(net, bus)

Automatically detect substations, recreate the substation DataFrame
=======

In [5]:
df_elements = get_all_element_terminals_df(net)
df_elements_plus_switches = add_switch_info_to_elements_df(net, df_elements)
df_final = pivot_to_connected_buses(df_elements_plus_switches)
df_final = add_bus_coupler_columns(net, df_final)
df_final = reorder_substation_df(df_final)

# The previous lines can be performed all at once using the following call:
# df_final = detect_substations(net)

In [6]:
df_final

,bus_0,bus_1,bus_2,b01_switch,b02_switch,b12_switch,connected_buses,n_busbars_in_substation,element_type,connected_elements,b0_switches,b1_switches,b2_switches
0,1,14,15,0,1,2,"[16, 17, 18, 19, 20, 21, 22, 23]",3,"[line, line, line, line, load, gen, trafo<PST>...","[0, 2, 3, 4, 0, 0, 5, 5]","[3, 6, 9, 12, 15, 18, 21, 24]","[4, 7, 10, 13, 16, 19, 22, 25]","[5, 8, 11, 14, 17, 20, 23, 26]"
1,2,24,<NA>,27,<NA>,<NA>,"[25, 26, 27, 28]",2,"[line, line, load, gen]","[2, 5, 1, 1]","[28, 30, 32, 34]","[29, 31, 33, 35]",NaN
2,3,29,<NA>,36,<NA>,<NA>,"[30, 31, 32, 33, 34, 35]",2,"[line, line, line, trafo, trafo, load]","[3, 5, 6, 0, 1, 2]","[37, 39, 41, 43, 45, 47]","[38, 40, 42, 44, 46, 48]",NaN
3,4,36,<NA>,49,<NA>,<NA>,"[37, 38, 39, 40, 41]",2,"[line, line, line, trafo, load]","[1, 4, 6, 2, 3]","[50, 52, 54, 56, 58]","[51, 53, 55, 57, 59]",NaN
4,5,42,<NA>,60,<NA>,<NA>,"[43, 44, 45, 46, 47, 48]",2,"[line, line, line, trafo, load, gen]","[7, 8, 9, 2, 4, 2]","[61, 63, 65, 67, 69, 71]","[62, 64, 66, 68, 70, 72]",NaN
5,8,49,<NA>,73,<NA>,<NA>,"[50, 51, 52, 53, 54]",2,"[line, line, trafo, trafo, load]","[10, 11, 1, 4, 5]","[74, 76, 78, 80, 82]","[75, 77, 79, 81, 83]",NaN
6,12,55,<NA>,84,<NA>,<NA>,"[56, 57, 58, 59]",2,"[line, line, line, load]","[9, 13, 14, 9]","[85, 87, 89, 91]","[86, 88, 90, 92]",NaN


In [7]:
net.multi_bb_substation

,bus_0,bus_1,bus_2,b01_switch,b02_switch,b12_switch,connected_buses,n_busbars_in_substation,element_type,connected_elements,b0_switches,b1_switches,b2_switches
0,1,14,15,0,1,2,"[16, 17, 18, 19, 20, 21, 22, 23]",3,"[line, line, line, line, load, gen, trafo<PST>...","[0, 2, 3, 4, 0, 0, 5, 5]","[3, 6, 9, 12, 15, 18, 21, 24]","[4, 7, 10, 13, 16, 19, 22, 25]","[5, 8, 11, 14, 17, 20, 23, 26]"
1,2,24,<NA>,27,<NA>,<NA>,"[25, 26, 27, 28]",2,"[line, line, load, gen]","[2, 5, 1, 1]","[28, 30, 32, 34]","[29, 31, 33, 35]",NaN
2,3,29,<NA>,36,<NA>,<NA>,"[30, 31, 32, 33, 34, 35]",2,"[line, line, line, trafo, trafo, load]","[3, 5, 6, 0, 1, 2]","[37, 39, 41, 43, 45, 47]","[38, 40, 42, 44, 46, 48]",NaN
3,4,36,<NA>,49,<NA>,<NA>,"[37, 38, 39, 40, 41]",2,"[line, line, line, trafo, load]","[1, 4, 6, 2, 3]","[50, 52, 54, 56, 58]","[51, 53, 55, 57, 59]",NaN
4,5,42,<NA>,60,<NA>,<NA>,"[43, 44, 45, 46, 47, 48]",2,"[line, line, line, trafo, load, gen]","[7, 8, 9, 2, 4, 2]","[61, 63, 65, 67, 69, 71]","[62, 64, 66, 68, 70, 72]",NaN
5,8,49,<NA>,73,<NA>,<NA>,"[50, 51, 52, 53, 54]",2,"[line, line, trafo, trafo, load]","[10, 11, 1, 4, 5]","[74, 76, 78, 80, 82]","[75, 77, 79, 81, 83]",NaN
6,12,55,<NA>,84,<NA>,<NA>,"[56, 57, 58, 59]",2,"[line, line, line, load]","[9, 13, 14, 9]","[85, 87, 89, 91]","[86, 88, 90, 92]",NaN
